# Pipeline Track C — Embedding-Based (v2) — Tim apace

## Arsitektur v2 (embedding-based)
- **Bukan lagi** load ConvNeXt `.pt` → forward image
- **Sekarang:** load embedding cache `.npy` → fit KNN/linear head → predict
- Semua inference berjalan di **CPU dalam hitungan menit**
- Tidak butuh GPU untuk Track C!

## Urutan wajib
1. Mount Drive → Clone repo → Install deps
2. Verifikasi semua path (OOF, embedding test, folds_v2.csv)
3. Validasi OOF + Nested validation (estimasi JUJUR)
4. Threshold tuning
5. Ensemble inference dari embedding
6. Generate + Validator + Manifest → UNGGAH

> ⚠️ **Jalankan setiap cell berurutan. Jangan skip Cell 4 (verifikasi path)!**

In [ ]:
# CELL 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# CELL 2: Clone repo ke path tetap (tidak dobel kalau sudah ada)
import os, sys

REPO_DIR = '/content/satria-data-bdcugm02'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone https://github.com/agaggigit/satria-data-bdcugm02.git {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} pull')

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Tambahkan track_b/src ke path agar heads.py bisa di-import
track_b_src = os.path.join(REPO_DIR, 'track_b', 'src')
if track_b_src not in sys.path:
    sys.path.insert(0, track_b_src)

os.chdir(REPO_DIR)
print(f'CWD: {os.getcwd()}')
print(f'sys.path[0]: {sys.path[0]}')

In [ ]:
# CELL 3: Install dependencies
# timm tidak diperlukan lagi untuk inference (embedding-based)
# tapi scikit-learn diperlukan untuk KNN/linear head
import os
os.system('pip install -q scikit-learn pandas numpy lightgbm')
print('Dependencies OK')

In [ ]:
# CELL 4: Verifikasi semua path sebelum pipeline
# =======================================================
# SESUAIKAN di sini jika path Drive berbeda!
# Kamu bisa override cfg sebelum dipakai pipeline.
# =======================================================
import os, sys
sys.path.insert(0, '/content/satria-data-bdcugm02')
sys.path.insert(0, '/content/satria-data-bdcugm02/track_b/src')

from track_c.src.config_c import CFG_C, COMPOSITIONS

print("=== VERIFIKASI PATH ===")
all_ok = True

# Path dasar
paths_to_check = {
    'folds_v2.csv (Track A, data bersih)': CFG_C.folds_csv,
    'submission.csv (template panitia)': CFG_C.sample_sub_path,
    'output_dir Track C': CFG_C.output_dir,
}
for name, path in paths_to_check.items():
    exists = os.path.exists(path)
    status = 'OK' if exists else 'TIDAK DITEMUKAN'
    print(f'  [{status}] {name}: {path}')
    if not exists:
        all_ok = False

# Path per komposisi
print()
print("=== ARTEFAK KOMPOSISI ===")
for key, comp in COMPOSITIONS.items():
    oof_ok = os.path.exists(comp.oof)
    emb_ok = os.path.exists(comp.emb_test)
    print(f'  Komposisi [{key}]:')
    print(f'    OOF      [{"OK" if oof_ok else "TIDAK ADA"}]: {comp.oof}')
    print(f'    emb_test [{"OK" if emb_ok else "TIDAK ADA"}]: {comp.emb_test}')
    print(f'    Head     : {comp.head}  |  Backbone: {comp.backbone}')
    if not oof_ok or not emb_ok:
        all_ok = False

print()
print(f'Komposisi AKTIF: {CFG_C.active_compositions}')
print()
if all_ok:
    print('✅ Semua path ditemukan. Lanjutkan ke Cell 5.')
else:
    print('❌ Ada path yang belum siap. Perbaiki config_c.py sebelum lanjut!')

In [ ]:
# CELL 5: Validasi OOF + Nested Validation
# Nested validation WAJIB dijalankan sebelum commit ke threshold.
# Ini memberikan estimasi JUJUR seberapa gain yang akan bertahan di test.
import numpy as np
import pandas as pd
from track_c.src.config_c import CFG_C
from track_c.src.nested_validation import nested_cv_threshold

folds_df = pd.read_csv(CFG_C.folds_csv)
print(f'folds_v2.csv: {len(folds_df)} baris')
print(f'Distribusi label: {dict(folds_df["label"].value_counts().sort_index())}')

# Load OOF dari komposisi aktif
active_key = CFG_C.active_compositions[0]
oof_path = CFG_C.compositions[active_key].oof
oof = np.load(oof_path)
print(f'\nOOF shape: {oof.shape}')
assert oof.shape == (len(folds_df), 3), f'Shape mismatch!'
assert not np.isnan(oof).any(), 'OOF mengandung NaN!'
assert np.allclose(oof.sum(axis=1), 1.0, atol=1e-3), 'OOF prob tidak sum ke 1!'
print('✅ OOF valid')

# Nested validation
nested_result = nested_cv_threshold(
    oof_probs=oof,
    true_labels=folds_df['label'].values,
    fold_assignments=folds_df['fold'].values,
    n_steps=30,   # langkah ≈ 0.05
    verbose=True,
)

print(f"\n📊 RINGKASAN:")
print(f"  Nested CV mean  : {nested_result['nested_mean']:.5f}")
print(f"  Nested CV min   : {nested_result['nested_min']:.5f}")
print(f"  Tuning-penuh F1 : {nested_result['full_tuned_f1']:.5f}")
print(f"  Illusion gap    : {nested_result['illusion_gap']:+.5f}")
print()
print("💡 Angka yang kamu laporkan ke tim = nested_mean, BUKAN tuning-penuh!")

In [ ]:
# CELL 6: Threshold Tuning (di seluruh OOF, untuk dipakai ke test)
from track_c.src.threshold_tuning import tune_thresholds_oof

best_thresholds = tune_thresholds_oof(
    oof_probs=oof,
    true_labels=folds_df['label'].values,
    n_steps=30,
)

print(f'\nThreshold final: {best_thresholds}')
print('Simpan angka ini untuk manifest!')

In [ ]:
# CELL 7: Ensemble Inference dari Embedding Cache
# Semua berjalan di CPU — tidak butuh GPU
from track_c.src.ensemble_embedding import run_ensemble_inference

test_probs = run_ensemble_inference(
    cfg=CFG_C,
    folds_df=folds_df,
    ensemble_weights=None,   # None = bobot seragam
)

print(f'test_probs shape: {test_probs.shape}')   # harus (1458, 3)
assert test_probs.shape[0] == 1458, f'Shape salah! Dapat {test_probs.shape}'
assert not np.isnan(test_probs).any(), 'test_probs mengandung NaN!'
print('✅ Inference berhasil')

In [ ]:
# CELL 8: Apply Threshold + Generate Submission + Validator + Manifest
# Ini adalah pipeline lengkap yang menggabungkan semua langkah
from track_c.src.threshold_tuning import apply_thresholds
from track_c.src.validator import validate_submission
from track_c.src.manifest import build_manifest, save_manifest, print_manifest_summary
import pandas as pd
import os

SUBMISSION_NUMBER = 2   # ← GANTI sesuai submission yang dikirim (1/2/3)
NOTES = "SigLIP2-SO400M KNN, threshold tuning, folds_v2.csv (data bersih)"  # ← isi catatan

# Apply threshold
final_labels = apply_thresholds(test_probs, best_thresholds)
label_counts = dict(zip(*np.unique(final_labels, return_counts=True)))
print(f'Distribusi prediksi: {label_counts}')

# Generate CSV
template_df = pd.read_csv(CFG_C.sample_sub_path)
sub_df = template_df.copy()
sub_df['predicted'] = final_labels.astype(int)

team_suffix = f'_{CFG_C.team_name}' if CFG_C.team_name else ''
submission_filename = f'submission{team_suffix}.csv'
os.makedirs(CFG_C.output_dir, exist_ok=True)
submission_path = os.path.join(CFG_C.output_dir, submission_filename)
sub_df.to_csv(submission_path, index=False)
print(f'Submission file: {submission_path}')

# Validator
is_valid = validate_submission(submission_path, CFG_C.sample_sub_path)
if not is_valid:
    raise RuntimeError('❌ SUBMISSION TIDAK VALID! Jangan upload sebelum diperbaiki!')

# Manifest
manifest = build_manifest(
    submission_number=SUBMISSION_NUMBER,
    submission_path=submission_path,
    cfg=CFG_C,
    active_compositions=CFG_C.active_compositions,
    thresholds=best_thresholds,
    nested_result=nested_result,
    notes=NOTES,
)
save_manifest(manifest, CFG_C.output_dir, SUBMISSION_NUMBER)
print_manifest_summary(manifest)

print()
print('='*55)
print(f'✅ Submission {SUBMISSION_NUMBER} SIAP diunggah ke portal panitia!')
print(f'   File: {submission_path}')
print('   ⚠️  Ingat: unggah ≤ 28 Juli (panen) / ≤ 29 Juli (asuransi)')
print('   ⚠️  Tie-break: yang unggah LEBIH DULU menang kalau skor sama!')
print('='*55)

## (OPSIONAL) Weight Search Multi-Komposisi

Jalankan cell ini **hanya kalau Track B sudah menyerahkan OOF dari backbone kedua/ketiga.**

Kalau masih 1 komposisi → skip cell ini.

In [ ]:
# CELL 9 (OPSIONAL): Weight search multi-komposisi
# Jalankan setelah OOF backbone kedua tersedia
# Hapus tanda # di bawah untuk mengaktifkan

# from track_c.src.nested_validation import nested_cv_weight_search
# 
# # Load semua OOF
# oof_comp2_path = CFG_C.compositions["dinov3_linear"].oof  # ganti key sesuai komposisi
# oof_comp2 = np.load(oof_comp2_path)
# 
# list_oofs = [oof, oof_comp2]   # urutan harus sama dengan active_compositions
# 
# result_weight = nested_cv_weight_search(
#     list_oof_probs=list_oofs,
#     true_labels=folds_df['label'].values,
#     fold_assignments=folds_df['fold'].values,
#     n_simplex=200,   # batasi untuk menghindari overfit
#     verbose=True,
# )
# 
# print(f"Best weights: {result_weight['best_weights']}")
# print(f"Nested mean  : {result_weight['nested_mean']:.5f}")
# print(f"Illusion gap : {result_weight['illusion_gap']:+.5f}")

print('(Cell ini opsional — skip kalau masih 1 komposisi)')

## Checklist Sebelum Upload

- [ ] Validator lolos (hijau di Cell 8)
- [ ] Manifest tersimpan di output_dir
- [ ] Nested CV mean sudah dicatat
- [ ] Catat aturan Submission 3 SEBELUM skor Submission 2 keluar
- [ ] Unggah ≤ 28 Juli (panen) atau ≤ 29 Juli (asuransi)
- [ ] ⚠️ Tie-break: yang unggah **lebih dulu** menang!

File output ada di:
```
/content/drive/MyDrive/BDC2026apace/output_trackC/submission_apace.csv
/content/drive/MyDrive/BDC2026apace/output_trackC/manifest_sub{N}.json
```